# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
# Capstone setup — fixed seed and safe paths.
from pathlib import Path
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "is_declining_label"
DATA_FILE = "content_refresh_anonymized.csv"

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
repo_root = next((p for p in search_roots if (p / "work" / "notebooks").is_dir()), None)
if repo_root is None:
    repo_root = cwd / "flyrank-ml-internship"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship.git", str(repo_root)],
            check=True,
        )

csv_candidates = [
    repo_root / "data" / "raw" / DATA_FILE,
    cwd / "data" / "raw" / DATA_FILE,
    cwd / "flyrank-ml-internship-starter" / "data" / "raw" / DATA_FILE,
]
csv_path = next((p for p in csv_candidates if p.is_file() and p.stat().st_size > 0), None)
if csv_path is None:
    starter_dir = cwd / "flyrank-ml-internship-starter"
    if not starter_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git", str(starter_dir)],
            check=True,
        )
    candidate = starter_dir / "data" / "raw" / DATA_FILE
    if candidate.is_file() and candidate.stat().st_size > 0:
        csv_path = candidate
if csv_path is None:
    raise FileNotFoundError("Could not locate the anonymized starter CSV. Follow SETUP.md first.")

df = pd.read_csv(csv_path, low_memory=False)
required = {
    "content_id", "client_id", "trend_direction", "days_since_last_update",
    "impressions_90d", "content_age_days", "word_count", "char_count",
    "search_volume", "competition", "cpc", "content_type", "main_intent",
    "competition_level",
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required columns missing: {missing}")
df[TARGET] = df["trend_direction"].eq("down").astype("int8")
print(f"Starter benchmark data: {len(df):,} rows × {df.shape[1]:,} source columns")
print(f"Observed decline-label rate: {df[TARGET].mean():.3f}")
print("Identifiers are grouping-only; private names, URLs, and queries are not displayed.")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
# Warehouse release profile copied from the executed full-release exploration notebook.
# This cell documents the release profile without triggering another 79M-row scan.
warehouse_profile = pd.DataFrame([
    {"table": "dim_clients", "rows": 104, "grain": "one pseudonymized client"},
    {"table": "dim_content", "rows": 519606, "grain": "one content item"},
    {"table": "fact_content_daily_performance", "rows": 78835655, "grain": "report date × client × content"},
    {"table": "fact_content_daily_performance_sample", "rows": 11694072, "grain": "same grain; June 2026 final month"},
    {"table": "fact_content_query_90d", "rows": 2414248, "grain": "client × content × query hash"},
])
display(warehouse_profile)
print("Release: build v20260703; daily history 2025-01-27 through 2026-06-30.")
print("Important: the June sample is the final outcome month and must not be used to design label logic.")
print("The headline model comparison below is from the 30,000-row starter slice, not a full-warehouse model.");
print("See notebooks/03_working_with_the_full_release.ipynb for the DuckDB/Hugging Face scan and its feature prototype.")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
# Method: observed label, leakage-aware features, client-grouped holdout, and baseline.
FEATURES = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc", "content_type", "main_intent", "competition_level",
]
NUMERIC_FEATURES = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc",
]
CATEGORICAL_FEATURES = ["content_type", "main_intent", "competition_level"]

# Never use target/trend-derived columns, IDs, current traffic totals, or product flags as model features.
forbidden = {
    "content_id", "client_id", TARGET, "trend_direction", "trend_pct",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "health_score", "needs_ctr_fix", "is_quick_win",
    "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d",
    "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
}
assert not (set(FEATURES) & forbidden), "A forbidden or leakage-prone field entered the feature list."

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
groups = df["client_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
train_rows, test_rows = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
model_probability = model.predict_proba(X_test)[:, 1]
model_order = np.argsort(-model_probability, kind="stable")

# Transparent Week-4 rule: >=180 days since update AND >=500 trailing-90-day impressions.
baseline_selected = (
    pd.to_numeric(test_rows["days_since_last_update"], errors="coerce").ge(180).fillna(False)
    & pd.to_numeric(test_rows["impressions_90d"], errors="coerce").ge(500).fillna(False)
)
baseline_score = np.where(
    baseline_selected,
    pd.to_numeric(test_rows["impressions_90d"], errors="coerce").fillna(0).to_numpy(),
    0.0,
)
baseline_frame = pd.DataFrame({
    "row_position": np.arange(len(test_rows)),
    "score": baseline_score,
    "impressions": pd.to_numeric(test_rows["impressions_90d"], errors="coerce").fillna(0).to_numpy(),
    "age": pd.to_numeric(test_rows["days_since_last_update"], errors="coerce").fillna(0).to_numpy(),
})
baseline_order = baseline_frame.sort_values(
    ["score", "impressions", "age"], ascending=[False, False, False], kind="mergesort"
)["row_position"].to_numpy()

def precision_at_k(order, labels, k):
    labels = np.asarray(labels)
    k = min(k, len(labels))
    return float(labels[np.asarray(order)[:k]].mean()) if k else float("nan")

print(f"Train: {len(train_idx):,} rows / {len(set(groups.iloc[train_idx]))} clients")
print(f"Test:  {len(test_idx):,} rows / {len(set(groups.iloc[test_idx]))} clients")
print(f"Test observed-decline base rate: {y_test.mean():.3f}")
print("Client overlap: 0")
print("Leakage guard: target/trend fields, IDs, current traffic totals, and product flags are excluded.")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# Results: both rankings evaluated on exactly the same held-out rows.
y_test_array = y_test.to_numpy()
baseline_ap = average_precision_score(y_test_array, np.argsort(np.argsort(-baseline_score, kind="stable"), kind="stable"))
model_ap = average_precision_score(y_test_array, model_probability)

results = pd.DataFrame([
    {
        "method": "Week-4 rule baseline",
        "test_base_rate": y_test.mean(),
        "average_precision": baseline_ap,
        "precision_at_20": precision_at_k(baseline_order, y_test_array, 20),
        "precision_at_50": precision_at_k(baseline_order, y_test_array, 50),
    },
    {
        "method": "Logistic Regression",
        "test_base_rate": y_test.mean(),
        "average_precision": model_ap,
        "precision_at_20": precision_at_k(model_order, y_test_array, 20),
        "precision_at_50": precision_at_k(model_order, y_test_array, 50),
    },
])
display(results.round(3))
print("Interpretation: metrics measure alignment with the observed decline label, not the effect of refreshing a page.")
print("Compare only methods on the same test rows. This is the starter-slice benchmark, not a model trained on all 78.8M daily rows.")

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = results.set_index("method")[["precision_at_20", "precision_at_50"]]
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Top-of-list precision on the same client-held-out split")
ax.set_ylabel("Share with observed decline label")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(["Precision@20", "Precision@50"], frameon=False)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

## 5. Limitations

*What this work cannot claim.*

In [ ]:
# Limitations — explicit and public-safe.
limitations = [
    "The headline model benchmark uses the 30,000-row starter snapshot, not a model trained on the full warehouse.",
    "The target is an observed trend label (trend_direction == 'down'), not a confirmed future outcome or a refresh-success label.",
    "The split holds out clients, but it is not a time-forward validation; unobserved seasonality and later distribution shifts remain.",
    "The rule baseline uses freshness and 90-day impressions, while the model uses a different feature set; the comparison is operational, not equal-information.",
    "The full-release prototype in notebooks/03 used outcome-window position volatility and fixed-window query signals that may overlap the label window; those metrics are not used as headline evidence.",
    "A static snapshot cannot show that refreshing a page causes improvement. Every ranked item requires human review.",
    "Some clients have limited history and GA4 coverage differs by client; zero-filled unavailable data must not be treated as true zero activity.",
]
for item in limitations:
    print("•", item)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Ranked recommendations — build a safe human-review queue from transparent signals only.
queue = df[["content_id", "client_id", "days_since_last_update", "impressions_90d"]].copy()
queue["days_since_last_update"] = pd.to_numeric(queue["days_since_last_update"], errors="coerce")
queue["impressions_90d"] = pd.to_numeric(queue["impressions_90d"], errors="coerce")
age_ok = queue["days_since_last_update"].notna() & queue["days_since_last_update"].ge(0)
impr_ok = queue["impressions_90d"].notna() & queue["impressions_90d"].ge(0)

queue["reason_code"] = np.select(
    [
        ~(age_ok & impr_ok),
        age_ok & impr_ok & queue["days_since_last_update"].ge(180) & queue["impressions_90d"].ge(500),
        age_ok & impr_ok & queue["days_since_last_update"].lt(180) & queue["impressions_90d"].ge(500),
        age_ok & impr_ok & queue["days_since_last_update"].ge(180) & queue["impressions_90d"].lt(500),
    ],
    [
        "missing_signal_manual_check",
        "stale_visible_review",
        "visible_recent_review",
        "stale_low_visibility_check",
    ],
    default="routine_monitor",
)
queue["priority_tier"] = queue["reason_code"].map({
    "stale_visible_review": 1,
    "visible_recent_review": 2,
    "stale_low_visibility_check": 3,
    "missing_signal_manual_check": 4,
    "routine_monitor": 5,
}).astype(int)
queue = queue.sort_values(
    ["priority_tier", "impressions_90d", "days_since_last_update"],
    ascending=[True, False, False], na_position="last", kind="mergesort"
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
display(queue["reason_code"].value_counts().rename_axis("reason_code").reset_index(name="rows"))
print("Priority 1: stale (>=180 days) and visible (>=500 impressions/90 days) — review first.")
print("Do not print or export raw page URLs, names, or search queries. Human review is mandatory.")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# Artifacts: save only aggregate charts/receipts and print the ML-12 communication cuts.
figures_dir = repo_root / "work" / "figures"
outputs_dir = repo_root / "work" / "outputs"
figures_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

# Save a public-safe chart from the aggregate results only.
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = results.set_index("method")[["precision_at_20", "precision_at_50"]]
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Model vs rule baseline — client-held-out starter-slice test")
ax.set_ylabel("Precision among top-ranked pages")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(["Precision@20", "Precision@50"], frameon=False)
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(figures_dir / "capstone_model_vs_baseline.png", dpi=160, bbox_inches="tight")
plt.close(fig)

summary = {
    "data_scope": "30,000-row anonymized starter slice for the headline benchmark",
    "warehouse_profile": {
        "fact_content_daily_performance_rows": 78835655,
        "date_start": "2025-01-27",
        "date_end": "2026-06-30",
        "release": "build v20260703",
    },
    "random_state": RANDOM_STATE,
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "train_clients": int(len(set(groups.iloc[train_idx]))),
    "test_clients": int(len(set(groups.iloc[test_idx]))),
    "test_base_rate": float(y_test.mean()),
    "results": results.round(6).to_dict(orient="records"),
    "limitations": "not a time-forward forecast; no causal refresh claim; model benchmark uses starter slice",
}
(outputs_dir / "capstone_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved aggregate chart:", figures_dir / "capstone_model_vs_baseline.png")
print("Saved metrics receipt:", outputs_dir / "capstone_results.json")
print("\n5-minute demo outline:")
print("1. State the decision: which content item should a person review first?")
print("2. Show the warehouse grain/date window and explain the starter-slice benchmark distinction.")
print("3. Compare the Week-4 rule and Logistic Regression on the same client-held-out test set.")
print("4. Explain the label, leakage exclusions, and why the model is not a causal refresh test.")
print("5. Show the ranked reason-code playbook and the human-review safeguards.")
print("\nSocial-post cut:")
print("I compared a transparent content-refresh rule with Logistic Regression on an anonymized starter snapshot using a client-held-out split. The model had higher measured top-20 and top-50 precision on this split, but the target is an observed trend label—not proof that refreshing a page improves outcomes. The paper documents the data scope, leakage checks, limitations, and human-review playbook.")
print("\nEmployer-facing summary:")
print("I built a reproducible content-review research workflow using FlyRank's anonymized data resources. I compared a transparent rule baseline with Logistic Regression on the same client-held-out starter-slice test rows, with explicit leakage checks and public-safe outputs. The result is a human-review aid; full-warehouse model claims are not made from the current benchmark.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
